# Run a Registered Workflow

Run `qdash config init --project-id your-project-id` in a terminal, then fill in `api_token` in the generated file. Use `--base-url` for a remote deployment. If the file already exists, edit it instead. `QDash()` reads its `default_profile`; use `QDash.from_profile("local")` to select a profile explicitly. See the [Python reference](../../../docs/reference/clients/python.md#connect-to-a-deployment) for setup. The workflow must already be saved and deployed in QDash, with its worker available.

In [ ]:
from qdash.sdk import QDash

qdash = QDash()

## List Workflows

In [ ]:
for workflow in qdash.workflows.list():
    print(workflow.name, workflow.chip_id)

## Select Workflow

Use a name from the list. Review its target chip and saved defaults before running.

In [ ]:
workflow = qdash.workflows.get("YOUR_WORKFLOW_NAME")
print(workflow.description)
print("Chip:", workflow.chip_id)
workflow.default_parameters

## Set Parameters

An empty dictionary uses saved defaults. Add only parameters supported by this workflow; saved defaults are not a complete parameter schema.

In [ ]:
parameters = {}

## Run Workflow

This cell starts the saved deployment on hardware. Re-running it starts another run. If the launch response is lost, check QDash before submitting again.

In [ ]:
run = workflow.run(parameters=parameters)
print("Execution:", run.execution_id)

## Wait for Completion

Wait for the whole workflow, not just its first execution. A timeout or interrupting this cell does not cancel the remote workflow; run this cell again to continue waiting, or use the optional cancellation cell below. This requires a server with the calibration run state endpoint. Progress is displayed by default; pass `progress=False` to silence it.

In [ ]:
result = run.wait(timeout_seconds=600)
print(result.status, "Succeeded:", result.succeeded)

## Cancel the Workflow (Optional)

To stop an active workflow, interrupt the wait cell first, then uncomment and run the cell below. Keep the existing `run`; do not re-run the launch cell. Cancellation requires editor access to the project and a worker able to process it.

`cancel()` requests cancellation of the whole workflow; `wait()` confirms its final state. If the request loses its connection, check `run.get().status` before retrying. If waiting times out, repeat only the wait. See [the cancellation example](07_cancel_workflow.ipynb) for error handling and troubleshooting.

In [ ]:
# from qdash.sdk import QDashConflictError
#
# if not run.get().done:
#     try:
#         cancellation = run.cancel()
#         print(cancellation.status, cancellation.message)
#     except QDashConflictError:
#         print("Cancellation was refused; current state:", run.get().status)
# result = run.wait(timeout_seconds=600)
# print("Final state:", result.status, "Cancelled:", result.status == "cancelled")

## Inspect Results

In [ ]:
for execution in run.executions():
    print(execution.execution_id, execution.status)
    for task in execution.tasks:
        print(task.task_id, task.name, task.qid, task.status)